# Pydantic 运行时强校验
## BaseModel
所有 Pydantic 模型的基类。继承它之后，类就具备了解析字典/JSON、类型转换、自动校验等能力。如果传入的数据类型不符，它会尝试合理转换（例如字符串 "123" 转为 int 123），如果转换失败则抛出 ValidationError。

- model_validate(obj)：从任意 Python 对象（或字典）构造模型实例。

- model_validate_json(json_str)：直接从 JSON 字符串反序列化，性能显著优于 json.loads 后再验证。

- model_dump()：将模型对象递归转换为纯 Python 字典（支持 mode="json", include, exclude, exclude_unset 等过滤参数）。

- model_dump_json()：将模型直接序列化为紧凑的 JSON 格式字符串。

In [4]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    is_active: bool = True

user = User(id = "1", name = "alice")
print(user.model_dump())

{'id': 1, 'name': 'alice', 'is_active': True}


## Field(description=...)
用于给模型字段添加额外的元数据、校验规则或文档说明。Field 函数用于为模型字段绑定额外的验证规则和元数据。其中 description 属性主要用于 OpenAPI / JSON Schema 文档的自动生成，为 API 调用者提供字段语义解释。

In [5]:
from pydantic import BaseModel, Field

class Item(BaseModel):
    name: str = Field(description="商品的显示名称", min_length=2)
    price: float = Field(gt=0, description="商品的价格，必须大于0")

# @field_validator：字段级自定义校验逻辑
用于自定义某个字段的校验与格式化逻辑。@field_validator 取代了旧版的 @validator。它以类方法的形式修饰校验函数，用于实现复杂的业务逻辑拦截与字段值标准化。
- 通常定义为 @classmethod。

- 必须接收当前字段值 v，返回校验或修改后的值；若不合法，则直接 raise ValueError(...)。
- 通常定义为 mode="after"。（默认）：在 Pydantic 完成标准基础类型转换之后执行校验逻辑。此时传入的 v 已经是目标类型的对象。

- 必须接收当前字段值 mode="before"，返回校验或修改后的值；若不合法，则直接 ：在 Pydantic 尝试类型转换之前执行，。常用于预清洗脏数据（例如剥离字符串空白、清洗特殊字符、解析非标格式）。

In [10]:
from pydantic import BaseModel, field_validator

class RegisterForm(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def username_must_be_alphanumeric(cls, v: str) -> str:
        if not v.isalnum():
            raise ValueError("用户名必须是字母或数字")
        return v.lower()

In [12]:
from pydantic import BaseModel, Field, field_validator, ValidationInfo
from typing import Optional

class UserProfile(BaseModel):
    user_id: int = Field(
        ...,
        gt = 0,
        description = "用户ID，必须大于0"
    )
    user_name: str = Field(
        ...,
        min_length = 3,
        max_length = 20,
        description = "用户名，长度必须在3到20之间"
    )
    email: str = Field(
        ...,
        pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$",
        description = "用户邮箱，必须是有效的邮箱格式"
    )
    role: str = Field(
        default = 'user',
        description = "用户角色，默认为'user'，可选值为'user'或'admin'或'superadmin'"
    )

    # 1. 前置清晰，将传入的用户名清理
    @field_validator("user_name", mode = "before")
    @classmethod
    def strip_username(cls, v: str) -> str:
        if isinstance(v, str):
            return v.strip()
        return v

    # 2. 后置校验，业务逻辑拦截
    @field_validator("user_name", mode = "after")
    @classmethod
    def validate_forbidden_names(cls, v: str) -> str:
        forbidden = {"admin", "root", "superuser"}
        if v.lower() in forbidden:
            raise ValueError(f"用户名 '{v}' 是禁止使用的名称")
        return v

    # 3. 跨字段校验，验证管理员邮箱域名限制
    @field_validator('role', mode = 'after')
    @classmethod
    def check_admin_priviledge(cls, role_val: str, info: ValidationInfo) -> str:
        valid_roles = {"user", "admin", "superadmin"}
        if role_val not in valid_roles:
            raise ValueError(f"角色 '{role_val}' 无效，必须是 {valid_roles} 中的一个")

        # 从info.data获取已通过校验的邮箱字段
        email_val = info.data.get('email')
        if role_val in {"admin", "superadmin"} and email_val:
            if not email_val.endswith("@company.com"):
                raise ValueError("管理员和超级管理员的邮箱必须是 '@company.com' 域名")
        return role_val

# Pyhton异步编程
1. 定义协程函数
    - 使用`async def`声明的函数不再是常规函数，调用它不会立即执行内部代码，而是返回一个协程对象
    - 协程对象可暂停和恢复执行，必须通过事件循环驱动
2. await：暂停执行并让渡控制权
    await只能在async def内部使用，后面只能紧跟三类对象：
    1. 协程对象
    2. Task任务，对协程的封装
    3. Future，低级对象，代表某个异步操作的最终结果


## 并发控制和高级原语
